**Data:** bundled in `data/`. Any missing series is downloaded and cached on first run - no prerequisite notebook.


## Imports & data

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# Resolves against the repository, not the current working directory, so the
# notebook runs from anywhere. Cached CSVs are bundled; if one is missing it
# is downloaded once and cached.
import os, pathlib
import pandas as pd

REPO     = pathlib.Path.cwd()
while not (REPO / 'qm_strategy.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
DATA_DIR = REPO / 'data'
DATA_DIR.mkdir(exist_ok=True)


def load_daily(ticker, start=START, end=END):
    """Bundled CSV if present, otherwise fetch once and cache."""
    cache = DATA_DIR / f"{ticker}_{start}_{end}_daily.csv"
    if not cache.exists():
        import yfinance as yf
        print(f"{cache.name} not cached - downloading...")
        df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
        df.to_csv(cache)
    return pd.read_csv(cache, index_col=0, parse_dates=True)


spy = load_daily("SPY")
print(f"SPY: {len(spy)} rows, {spy.index[0].date()} to {spy.index[-1].date()}")


## Build the strategy

**The rule in plain English:**
- Each day, compute the 200-day simple moving average of the closing price
- If today's close is *above* the 200-day MA → be invested in SPY
- If today's close is *below* the 200-day MA → be in cash (earning nothing — we ignore money-market yield for simplicity)

**Commission model:**
We apply a round-trip cost every time we flip in or out of the market. A realistic retail commission today is near-zero per trade, but there's still bid-ask spread and market impact. We'll use **0.05% per trade** (5 basis points) — conservative but non-zero.

This matters more than it seems. If the strategy flips in and out 20 times over 5 years, that's 40 individual trades × 0.05% = 2% of capital gone to friction before we've earned a single dollar.

In [ ]:
COMMISSION = 0.0005   # 5 bps per trade (one-way)
MA_WINDOW  = 200      # days

# ── Compute the signal ───────────────────────────────────────────────────
spy['MA200']   = spy['Close'].rolling(MA_WINDOW).mean()
spy['Signal']  = (spy['Close'] > spy['MA200']).astype(int)  # 1 = invested, 0 = cash

# Signal is known at close, so we act on it the NEXT open.
# Shift by 1 so we never peek into the future.
spy['Position'] = spy['Signal'].shift(1).fillna(0)

# ── Detect trades (position changes) ────────────────────────────────────
spy['Trade']    = spy['Position'].diff().abs()   # 1 on entry or exit day
n_trades = int(spy['Trade'].sum())
print(f"Total trades (entries + exits): {n_trades}")
print(f"Round trips:                    {n_trades // 2}")
print(f"Total commission drag:          {n_trades * COMMISSION:.2%}")

## Compute daily returns

We calculate two return series in parallel:

- **Buy-and-hold:** simply the daily % change in SPY price every day
- **Strategy:** the daily % change *only on days we're invested*, minus commission on trade days

In [ ]:
spy['DailyReturn']  = spy['Close'].pct_change()

# Strategy return = daily return × position, minus commission on trade days
spy['StratReturn']  = (spy['DailyReturn'] * spy['Position']) - (spy['Trade'] * COMMISSION)

# Cumulative equity curves (start at 1.0 = $1 invested)
spy['BnH_Equity']   = (1 + spy['DailyReturn']).cumprod()
spy['Strat_Equity'] = (1 + spy['StratReturn']).cumprod()

# Trim NaNs from the MA warm-up period
result = spy.dropna(subset=['MA200']).copy()
print(f"Analysis window: {result.index[0].date()} → {result.index[-1].date()}")
print(f"({len(result)} trading days after 200-day MA warm-up)")

## Performance metrics

The five numbers that actually matter — each one tells a different part of the story.

In [ ]:
def compute_metrics(equity_curve, daily_returns, label, periods_per_year=252):
    total_ret    = equity_curve.iloc[-1] - 1
    n_years      = len(equity_curve) / periods_per_year
    ann_ret      = (equity_curve.iloc[-1]) ** (1 / n_years) - 1
    ann_vol      = daily_returns.std() * np.sqrt(periods_per_year)
    sharpe       = ann_ret / ann_vol if ann_vol > 0 else 0
    rolling_max  = equity_curve.cummax()
    drawdown     = (equity_curve - rolling_max) / rolling_max
    max_dd       = drawdown.min()
    pct_invested = (result['Position'] == 1).mean() if 'Position' in result.columns else 1.0

    return {
        'Strategy':       label,
        'Total Return':   f"{total_ret:.1%}",
        'Ann. Return':    f"{ann_ret:.1%}",
        'Ann. Volatility':f"{ann_vol:.1%}",
        'Sharpe Ratio':   f"{sharpe:.2f}",
        'Max Drawdown':   f"{max_dd:.1%}",
        '% Time Invested':f"{pct_invested:.0%}",
    }

bnh_metrics   = compute_metrics(result['BnH_Equity'],   result['DailyReturn'], 'Buy & Hold SPY')
strat_metrics = compute_metrics(result['Strat_Equity'],  result['StratReturn'], '200-day MA Strategy')
# % invested not meaningful for B&H — always 100%
bnh_metrics['% Time Invested'] = '100%'

pd.DataFrame([bnh_metrics, strat_metrics]).set_index('Strategy')

**Read this table slowly** before looking at any chart.

A few things to look for:

- Does the strategy's **Sharpe ratio** beat buy-and-hold? Sharpe = return per unit of risk. If the strategy has lower return AND lower Sharpe, it's worse in every dimension.
- The **% Time Invested** tells you how often the strategy was in the market. Less time invested means more time earning zero — that's a real cost even without commissions.
- The **Max Drawdown** might be better on the strategy (MA strategies do sometimes reduce drawdowns by going to cash during bad periods) even if total return is worse. That's the only genuine argument for this kind of strategy.

## Chart 1 — Equity curves

The cleanest summary: $1 invested at the start, how does each path look over time?

In [ ]:
fig = go.Figure()

start_price = result['Close'].iloc[0]

bnh_rebased   = result['BnH_Equity']   / result['BnH_Equity'].iloc[0]
strat_rebased = result['Strat_Equity'] / result['Strat_Equity'].iloc[0]
ma200_rebased = result['MA200']        / start_price  # same denominator as price

fig.add_trace(go.Scatter(
    x=result.index, y=bnh_rebased,
    name='Buy & Hold SPY',
    line=dict(color='#2E86AB', width=2.5)
))
fig.add_trace(go.Scatter(
    x=result.index, y=strat_rebased,
    name='200-day MA Strategy',
    line=dict(color='#E84855', width=2.5)
))
fig.add_trace(go.Scatter(
    x=result.index, y=ma200_rebased,
    name='200-day MA',
    line=dict(color='#F18F01', width=1.5, dash='dash'),
    opacity=0.7
))

in_cash = result['Position'] == 0
cash_starts = result.index[in_cash & (~in_cash.shift(1).fillna(False).astype(bool))]
cash_ends   = result.index[in_cash & (~in_cash.shift(-1).fillna(False).astype(bool))]
for s, e in zip(cash_starts, cash_ends):
    fig.add_vrect(x0=s, x1=e, fillcolor='rgba(200,200,200,0.25)', layer='below', line_width=0)

fig.add_annotation(
    x=0.01, y=0.97, xref='paper', yref='paper',
    text='<b>Grey shading</b> = strategy is in cash (not invested)',
    showarrow=False, font=dict(size=11, color='#666'), align='left'
)
fig.update_layout(
    title='Equity Curves: Buy & Hold vs 200-day MA Strategy (SPY)',
    yaxis_title='Portfolio Value (start = $1.00)',
    height=500, template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.85)
)
fig.show()

Look at the grey shaded regions — those are the periods the strategy was in cash. Notice whether they actually coincide with the *worst* parts of the SPY drawdown, or whether the strategy was late getting out and late getting back in. The latter is the classic failure mode of trend-following on indices: the signal lags, so you exit after the damage is done and re-enter after the recovery has started.

## Chart 2 — Price with MA200 and trade signals

This is the TradingView-style view: see exactly *when* signals fired relative to price. This is where the strategy's weaknesses become obvious to the eye.

In [ ]:
# Identify entry and exit points
entries = result[result['Trade'] == 1][result['Position'] == 1]
exits   = result[result['Trade'] == 1][result['Position'] == 0]

fig = go.Figure()

# Price + MA200
fig.add_trace(go.Scatter(
    x=result.index, y=result['Close'],
    name='SPY Close', line=dict(color='#2E86AB', width=1.5)
))
fig.add_trace(go.Scatter(
    x=result.index, y=result['MA200'],
    name='200-day MA', line=dict(color='#F18F01', width=2, dash='dot')
))

# Buy signals (green triangles up)
fig.add_trace(go.Scatter(
    x=entries.index, y=entries['Close'],
    mode='markers', name='Buy signal',
    marker=dict(symbol='triangle-up', size=10, color='#2DC653')
))

# Sell signals (red triangles down)
fig.add_trace(go.Scatter(
    x=exits.index, y=exits['Close'],
    mode='markers', name='Sell signal',
    marker=dict(symbol='triangle-down', size=10, color='#E84855')
))

fig.update_layout(
    title='SPY with 200-day MA — buy/sell signals',
    yaxis_title='Price (USD)',
    height=550,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(x=0.01, y=0.99)
)
fig.show()

Zoom into any sell signal and ask: **how much of the drop had already happened by the time the signal fired?** Because the MA lags price, the answer is almost always "most of it." Similarly, buy signals tend to fire after a significant recovery is already underway. You're selling near the bottom and buying near local highs — the exact opposite of what you'd want.

This isn't a flaw in *this particular implementation* — it's structural to any MA-based strategy. The smoothing that makes the MA robust also makes it slow.

## Chart 3 — Drawdown comparison

The strategy's one legitimate argument: does it reduce the *pain* of holding, even if it reduces returns?

In [ ]:
def rolling_drawdown(equity):
    return (equity - equity.cummax()) / equity.cummax() * 100

bnh_dd   = rolling_drawdown(result['BnH_Equity'])
strat_dd = rolling_drawdown(result['Strat_Equity'])

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=(
                        f'Buy & Hold SPY  (max drawdown: {bnh_dd.min():.1f}%)',
                        f'200-day MA Strategy  (max drawdown: {strat_dd.min():.1f}%)'
                    ),
                    vertical_spacing=0.1)

fig.add_trace(go.Scatter(x=bnh_dd.index, y=bnh_dd, fill='tozeroy',
                         name='B&H', line=dict(color='#2E86AB')), row=1, col=1)
fig.add_trace(go.Scatter(x=strat_dd.index, y=strat_dd, fill='tozeroy',
                         name='Strategy', line=dict(color='#E84855')), row=2, col=1)
fig.update_yaxes(title='Drawdown (%)', row=1, col=1)
fig.update_yaxes(title='Drawdown (%)', row=2, col=1)
fig.update_layout(height=600, showlegend=False, template='plotly_white',
                  title='Drawdown: Buy & Hold vs MA Strategy')
fig.show()

## Chart 4 — Rolling 1-year return comparison

Instead of one number, this shows *consistency*. A strategy that has a better 1-year return than buy-and-hold for most of the test period but a disastrous final year could still show a net loss. This chart reveals that temporal pattern.

In [ ]:
roll_bnh   = result['BnH_Equity'].pct_change(252) * 100
roll_strat = result['Strat_Equity'].pct_change(252) * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=roll_bnh.index, y=roll_bnh,
                         name='Buy & Hold SPY', line=dict(color='#2E86AB', width=2)))
fig.add_trace(go.Scatter(x=roll_strat.index, y=roll_strat,
                         name='200-day MA Strategy', line=dict(color='#E84855', width=2)))
fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1)

fig.update_layout(
    title='Rolling 1-year return: Buy & Hold vs MA Strategy',
    yaxis_title='1-year return (%)',
    height=450,
    template='plotly_white',
    hovermode='x unified',
)
fig.show()

## Chart 5 — Trade-by-trade attribution

This is the most surgical chart in the notebook. Each bar is one complete round trip (buy → sell), showing how much that specific trade contributed to or detracted from performance relative to just holding.

This is where you see **whipsaw** — trades that quickly enter and exit for small losses. Each one costs commission *twice* and earns almost nothing.

In [ ]:
# Build trade-by-trade log
entry_dates  = result.index[(result['Trade'] == 1) & (result['Position'] == 1)].tolist()
exit_dates   = result.index[(result['Trade'] == 1) & (result['Position'] == 0)].tolist()

# Pair them up (simple pairing — each entry followed by next exit)
trades = []
exit_q = list(exit_dates)
for entry in entry_dates:
    # find the first exit that comes after this entry
    future_exits = [e for e in exit_q if e > entry]
    if future_exits:
        exit_d  = future_exits[0]
        entry_p = result.loc[entry, 'Close']
        exit_p  = result.loc[exit_d, 'Close']
        trade_ret = (exit_p / entry_p - 1) - 2 * COMMISSION  # round-trip cost
        bnh_ret   = (exit_p / entry_p - 1)  # what B&H earned over same window
        alpha     = trade_ret - bnh_ret      # did we beat just holding through it?
        trades.append({
            'Entry': entry.date(),
            'Exit':  exit_d.date(),
            'Days':  (exit_d - entry).days,
            'Trade Return': f"{trade_ret:.2%}",
            'B&H Same Window': f"{bnh_ret:.2%}",
            'Alpha vs B&H': alpha,
        })
        exit_q.remove(exit_d)

trades_df = pd.DataFrame(trades)
print(f"Complete round trips: {len(trades_df)}")
trades_df

---
## Extension — Dual MA system (50/200)

You spotted the core problem with the single MA strategy: **the signal is too slow**. By the time the 200-day MA crosses, the damage is done on the way out and the recovery is already underway on the way back in.

The fix: **separate the entry signal from the exit signal.**

- **Entry:** price crosses above the 200-day MA (slow, deliberate — confirms a real uptrend)
- **Exit:** the 50-day MA crosses below the 200-day MA (faster — reacts to trend deterioration sooner)

This is the classic **50/200 Golden Cross / Death Cross** system. The entry is still patient, but now you have a faster tripwire for getting out.

The 50-day MA responds to roughly the last 10 weeks of price action. When it rolls over and crosses below the 200-day, you've seen enough deterioration to act — without needing price itself to make the full round-trip below the 200-day.

We run it through the same metrics and charts so the comparison is apples-to-apples.

In [ ]:
# ── Dual MA signal ────────────────────────────────────────────────────────
spy['MA50']  = spy['Close'].rolling(50).mean()

# In the market when BOTH: price > MA200 AND MA50 > MA200
dual_signal = ((spy['Close'] > spy['MA200']) & (spy['MA50'] > spy['MA200'])).astype(int)

spy['DualPosition'] = dual_signal.shift(1).fillna(0)
spy['DualTrade']    = spy['DualPosition'].diff().abs()

n_dual = int(spy['DualTrade'].sum())
print(f"Dual MA  — total trades: {n_dual}  |  round trips: {n_dual // 2}  |  commission drag: {n_dual * COMMISSION:.2%}")
print(f"Single MA — total trades: {n_trades}  |  round trips: {n_trades // 2}  |  commission drag: {n_trades * COMMISSION:.2%}")

In [ ]:
spy['DualReturn']  = (spy['DailyReturn'] * spy['DualPosition']) - (spy['DualTrade'] * COMMISSION)
spy['Dual_Equity'] = (1 + spy['DualReturn']).cumprod()

result = spy.dropna(subset=['MA200']).copy()

def compute_metrics_v2(equity, returns, position_col, label):
    total_ret = equity.iloc[-1] - 1
    n_years   = len(equity) / 252
    ann_ret   = equity.iloc[-1] ** (1 / n_years) - 1
    ann_vol   = returns.std() * np.sqrt(252)
    sharpe    = ann_ret / ann_vol if ann_vol > 0 else 0
    max_dd    = ((equity - equity.cummax()) / equity.cummax()).min()
    pct_in    = (result[position_col] == 1).mean() if position_col else 1.0
    return {
        'Strategy':        label,
        'Total Return':    f"{total_ret:.1%}",
        'Ann. Return':     f"{ann_ret:.1%}",
        'Ann. Volatility': f"{ann_vol:.1%}",
        'Sharpe':          f"{sharpe:.2f}",
        'Max Drawdown':    f"{max_dd:.1%}",
        '% Invested':      f"{pct_in:.0%}",
    }

m_bnh    = compute_metrics_v2(result['BnH_Equity'],   result['DailyReturn'], None,           'Buy & Hold SPY')
m_single = compute_metrics_v2(result['Strat_Equity'], result['StratReturn'], 'Position',     '200-day MA (single)')
m_dual   = compute_metrics_v2(result['Dual_Equity'],  result['DualReturn'],  'DualPosition', '50/200 Dual MA')
m_bnh['% Invested'] = '100%'

pd.DataFrame([m_bnh, m_single, m_dual]).set_index('Strategy')

### Equity curves — all three side by side

In [ ]:
start_price = result['Close'].iloc[0]

bnh_rebased   = result['BnH_Equity']   / result['BnH_Equity'].iloc[0]
strat_rebased = result['Strat_Equity'] / result['Strat_Equity'].iloc[0]
dual_rebased  = result['Dual_Equity']  / result['Dual_Equity'].iloc[0]
ma200_rebased = result['MA200'] / start_price
ma50_rebased  = result['MA50']  / start_price

fig = go.Figure()
fig.add_trace(go.Scatter(x=result.index, y=bnh_rebased,
                         name='Buy & Hold SPY', line=dict(color='#2E86AB', width=2.5)))
fig.add_trace(go.Scatter(x=result.index, y=strat_rebased,
                         name='200-day MA (single)', line=dict(color='#E84855', width=2)))
fig.add_trace(go.Scatter(x=result.index, y=dual_rebased,
                         name='50/200 Dual MA', line=dict(color='#2DC653', width=2.5)))
fig.add_trace(go.Scatter(x=result.index, y=ma200_rebased,
                         name='200-day MA', line=dict(color='#F18F01', width=1.5, dash='dash'), opacity=0.7))
fig.add_trace(go.Scatter(x=result.index, y=ma50_rebased,
                         name='50-day MA', line=dict(color='#A23B72', width=1.5, dash='dash'), opacity=0.7))

in_cash_dual = result['DualPosition'] == 0
cash_starts  = result.index[in_cash_dual & ~in_cash_dual.shift(1).fillna(False).astype(bool)]
cash_ends    = result.index[in_cash_dual & ~in_cash_dual.shift(-1).fillna(False).astype(bool)]
for s, e in zip(cash_starts, cash_ends):
    fig.add_vrect(x0=s, x1=e, fillcolor='rgba(200,200,200,0.2)', layer='below', line_width=0)

fig.add_annotation(x=0.01, y=0.97, xref='paper', yref='paper',
                   text='<b>Grey shading</b> = Dual MA is in cash',
                   showarrow=False, font=dict(size=11, color='#666'), align='left')
fig.update_layout(
    title='Equity Curves: B&H vs Single MA vs Dual MA (SPY)',
    yaxis_title='Portfolio Value (start = $1.00)',
    height=500, template='plotly_white', hovermode='x unified',
    legend=dict(x=0.01, y=0.85)
)
fig.show()

### Price view — both MAs with entry/exit signals

Watch where the sell triangle fires relative to the price drop vs where the single MA would have fired. The 50-day is closer to current price — it rolls over faster.

In [ ]:
dual_entries = result.index[(result['DualTrade'] == 1) & (result['DualPosition'] == 1)]
dual_exits   = result.index[(result['DualTrade'] == 1) & (result['DualPosition'] == 0)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=result.index, y=result['Close'],
                         name='SPY Close', line=dict(color='#2E86AB', width=1.5)))
fig.add_trace(go.Scatter(x=result.index, y=result['MA200'],
                         name='200-day MA', line=dict(color='#F18F01', width=2, dash='dot')))
fig.add_trace(go.Scatter(x=result.index, y=result['MA50'],
                         name='50-day MA', line=dict(color='#A23B72', width=2, dash='dash')))
fig.add_trace(go.Scatter(x=dual_entries, y=result.loc[dual_entries, 'Close'],
                         mode='markers', name='Buy (Golden Cross)',
                         marker=dict(symbol='triangle-up', size=12, color='#2DC653')))
fig.add_trace(go.Scatter(x=dual_exits, y=result.loc[dual_exits, 'Close'],
                         mode='markers', name='Sell (Death Cross)',
                         marker=dict(symbol='triangle-down', size=12, color='#E84855')))
fig.update_layout(
    title='SPY — 50/200 Golden Cross & Death Cross signals',
    yaxis_title='Price (USD)',
    height=550, template='plotly_white', hovermode='x unified',
    xaxis_rangeslider_visible=False
)
fig.show()

### What to look for

**Did the dual MA exit earlier?** The Death Cross sell triangle should appear higher on the chart (earlier in the selloff) than where the single MA would have fired.

**Did it reduce whipsaw?** Check the trade count — fewer trades means less commission drag and less noise.

**Did Sharpe improve?** That's the real question. Better return per unit of risk taken is a genuine improvement. Lower return AND lower volatility is just a slower version of the same problem.

**The honest ceiling:** even if dual MA improves on single MA, both are still fighting the same structural battle — MAs lag price, every cash period costs opportunity in a rising market, and SPY's upward drift is relentless. What you've built here is the *intuition* for why trend-following on indices is hard, not a solved strategy. That intuition is exactly what you need before layering in something more complex like SMC.

## The verdict

Let the numbers from the metrics table and the charts tell the story — don't try to spin it. There are four honest outcomes:

**Outcome A: Strategy clearly loses on all dimensions.** Lower return, lower Sharpe, similar or worse drawdown. Verdict: the timing added no value. The friction and the lag hurt more than the protection helped.

**Outcome B: Strategy loses on return but wins on drawdown.** Lower return *and* lower max drawdown. Verdict: the strategy is a smoother ride to a worse destination. Whether this is worth it depends on whether you can honestly say you would have panic-sold during the B&H drawdowns. Most people can't hold through -30% regardless of what they tell themselves in advance.

**Outcome C: Strategy wins on Sharpe but not raw return.** Same or slightly lower return, but better risk-adjusted. This is the most intellectually honest argument for the strategy. It means every unit of risk you took was better-compensated.

**Outcome D: Strategy wins outright.** Higher return *and* higher Sharpe. This would be surprising on SPY with a 200-day MA, especially post-2010. If you see this, check your implementation for a look-ahead bias before celebrating.

---

### Next notebook: `03_swing_points.ipynb`

Detecting local highs and lows algorithmically — the foundational building block of every SMC concept. Once you can reliably find swing points in code, you're one step away from detecting order blocks, BOS/CHoCH, and FVGs.